# Gold reports
Small, ready-to-use report tables built from the Gold star schema, for the Streamlit app and BI tools.
Each report = one SQL query → DataFrame → Delta table `workspace.sme_risk_gold.rpt_*` (overwritten every run).

In [0]:
from pyspark.sql import functions as F

CATALOG     = "workspace"
GOLD_SCHEMA = f"{CATALOG}.sme_risk_gold"

def write_report(df, name):
    """Save a report table (overwrite) and print its row count."""
    (df.write
       .mode("overwrite")
       .format("delta")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{GOLD_SCHEMA}.{name}"))
    print(f"{name}: {spark.table(f'{GOLD_SCHEMA}.{name}').count():,} rows written")

## rpt_key_metrics
One row per headline metric for the latest snapshot (the EDA measures report), in display order.
Used by the app's Overview page.

In [0]:
query = f"""
WITH snap AS (
    SELECT
        COUNT(*)                                        AS total_companies,
        SUM(in_population)                              AS in_population,
        SUM(is_active)                                  AS active,
        SUM(is_distressed)                              AS distressed,
        SUM(is_deregistered)                            AS deregistered,
        MAX(snapshot_date_key)                          AS snapshot_date_key
    FROM {GOLD_SCHEMA}.fct_company_snapshot
),
expo AS (
    SELECT
        SUM(exposure_years)                             AS company_years,
        SUM(deregistrations)                            AS deregistrations,
        MIN(window_start_date_key)                      AS window_start_key
    FROM {GOLD_SCHEMA}.fct_exposure
),
comp AS (
    SELECT
        AVG(CASE WHEN has_abn         THEN 1 ELSE 0 END) AS share_abn,
        AVG(CASE WHEN has_name_change THEN 1 ELSE 0 END) AS share_renamed,
        AVG(CASE WHEN is_acn_named    THEN 1 ELSE 0 END) AS share_numbered
    FROM {GOLD_SCHEMA}.dim_company
),
age AS (
    SELECT PERCENTILE_DISC(0.5) WITHIN GROUP (ORDER BY age_years) AS median_age
    FROM {GOLD_SCHEMA}.fct_company_snapshot
    WHERE is_active = 1
),
metrics AS (
    SELECT 1  AS metric_order, 'Total companies'                       AS metric_name, CAST(total_companies AS DOUBLE)                       AS metric_value, 'count'   AS unit FROM snap
    UNION ALL SELECT 2,  'Active companies',                        CAST(active AS DOUBLE),                                        'count'   FROM snap
    UNION ALL SELECT 3,  'Distressed companies (strike-off + administration)', CAST(distressed AS DOUBLE),                         'count'   FROM snap
    UNION ALL SELECT 4,  'Distress share of active companies',      100.0 * distressed / active,                                   'percent' FROM snap
    UNION ALL SELECT 5,  'Deregistered in the last 12 months',      CAST(deregistered AS DOUBLE),                                  'count'   FROM snap
    UNION ALL SELECT 6,  'Company-years at risk',                   company_years,                                                 'count'   FROM expo
    UNION ALL SELECT 7,  'Yearly deregistration probability',       100 * (1 - EXP(-deregistrations / company_years)),             'percent' FROM expo
    UNION ALL SELECT 8,  'Companies with an ABN',                   100 * share_abn,                                               'percent' FROM comp
    UNION ALL SELECT 9,  'Companies ever renamed',                  100 * share_renamed,                                           'percent' FROM comp
    UNION ALL SELECT 10, 'Numbered (ACN-named) companies',          100 * share_numbered,                                          'percent' FROM comp
    UNION ALL SELECT 11, 'Median age of active companies',          CAST(median_age AS DOUBLE),                                    'years'   FROM age
)
SELECT
    m.metric_order,
    m.metric_name,
    ROUND(m.metric_value, 2)                                       AS metric_value,
    m.unit,
    to_date(CAST(s.snapshot_date_key AS STRING), 'yyyyMMdd')       AS snapshot_date,
    to_date(CAST(e.window_start_key  AS STRING), 'yyyyMMdd')       AS window_start_date
FROM metrics m
CROSS JOIN snap s
CROSS JOIN expo e
ORDER BY m.metric_order
"""
df = spark.sql(query)
df.display()

In [0]:
write_report(df, "rpt_key_metrics")

In [0]:
%sql
SELECT
    COUNT(*)                         AS rows,             
    COUNT(DISTINCT metric_order)     AS distinct_orders,  
    COUNT_IF(metric_value IS NULL)   AS null_values,      
    MIN(snapshot_date)               AS snapshot_date,
    MIN(window_start_date)           AS window_start_date
FROM workspace.sme_risk_gold.rpt_key_metrics

## rpt_monthly_deregistrations
One row per month in the observation window: deregistrations in total, by segment group and by age at deregistration, with month-on-month change, running total and a flag for partial edge months.
Used by the app's trend chart (May 2026 batch spike).

In [0]:
query = f"""
WITH window_dates AS (
    SELECT
        to_date(CAST(MIN(window_start_date_key) AS STRING), 'yyyyMMdd') AS window_start,
        to_date(CAST(MAX(snapshot_date_key)     AS STRING), 'yyyyMMdd') AS window_end
    FROM {GOLD_SCHEMA}.fct_exposure
),
monthly AS (
    SELECT
        d.year_month,
        d.month_start_date,
        COUNT(*)                                                                    AS deregistrations,
        SUM(CASE WHEN k.borrower_segment = 'Trading company'  THEN 1 ELSE 0 END)    AS trading_companies,
        SUM(CASE WHEN k.borrower_segment <> 'Trading company' THEN 1 ELSE 0 END)    AS other_segments,
        SUM(CASE WHEN f.age_band_key IN (1, 2) THEN 1 ELSE 0 END)                   AS age_0_2,
        SUM(CASE WHEN f.age_band_key = 3       THEN 1 ELSE 0 END)                   AS age_2_3,
        SUM(CASE WHEN f.age_band_key IN (4, 5) THEN 1 ELSE 0 END)                   AS age_3_5,
        SUM(CASE WHEN f.age_band_key IN (6, 7) THEN 1 ELSE 0 END)                   AS age_5_plus,
        SUM(CASE WHEN f.age_band_key = 0       THEN 1 ELSE 0 END)                   AS age_unknown
    FROM {GOLD_SCHEMA}.fct_company_snapshot f
    JOIN {GOLD_SCHEMA}.dim_date d                   ON f.deregistration_date_key = d.date_key
    JOIN {GOLD_SCHEMA}.dim_company_classification k ON f.classification_key      = k.classification_key
    WHERE f.is_deregistered = 1
    GROUP BY d.year_month, d.month_start_date
)
SELECT
    m.year_month,
    m.month_start_date,
    m.deregistrations,
    m.trading_companies,
    m.other_segments,
    m.age_0_2,
    m.age_2_3,
    m.age_3_5,
    m.age_5_plus,
    m.age_unknown,
    m.deregistrations - LAG(m.deregistrations) OVER (ORDER BY m.month_start_date)   AS change_vs_previous_month,
    SUM(m.deregistrations) OVER (ORDER BY m.month_start_date)                       AS running_total,
    (m.month_start_date = trunc(w.window_start, 'MM') AND day(w.window_start) > 1)
      OR m.month_start_date = trunc(w.window_end, 'MM')                             AS is_partial_month
FROM monthly m
CROSS JOIN window_dates w
ORDER BY m.month_start_date
"""
df = spark.sql(query)
df.display()

In [0]:
write_report(df, "rpt_monthly_deregistrations")

In [0]:
%sql
SELECT
    COUNT(*)                                                            AS months,                 
    SUM(deregistrations)                                                AS total_deregistrations,  
    SUM(trading_companies + other_segments) - SUM(deregistrations)      AS segment_split_gap,    
    SUM(age_0_2 + age_2_3 + age_3_5 + age_5_plus + age_unknown)
      - SUM(deregistrations)                                            AS age_split_gap,          
    MAX(running_total)                                                  AS final_running_total,    
    COUNT_IF(is_partial_month)                                          AS partial_months          
FROM workspace.sme_risk_gold.rpt_monthly_deregistrations

## rpt_survival_curve
Survival to years 0–5 for all companies and trading companies, in two scenarios: all deregistrations, and excluding the May–June 2026 ASIC batch (sensitivity check).
Method: yearly probability q = 1 − exp(−deregistrations ÷ company-years); survival S(N) = exp(−Σ rate) = ∏(1 − q). Year 0 = 100%.
Used by the app's survival-curve page.

In [0]:
BATCH_MONTHS = "'2026-05', '2026-06'"   # deregistration batch excluded in the sensitivity scenario

query = f"""
WITH base AS (
    SELECT
        f.age_band_key,
        f.exposure_years,
        f.deregistrations,
        CASE WHEN f.deregistrations = 1 AND d.year_month IN ({BATCH_MONTHS}) THEN 0
             ELSE f.deregistrations END                                   AS deregistrations_excl_batch,
        k.borrower_segment
    FROM {GOLD_SCHEMA}.fct_exposure f
    JOIN {GOLD_SCHEMA}.fct_company_snapshot s       ON f.company_key            = s.company_key
    JOIN {GOLD_SCHEMA}.dim_date d                   ON s.deregistration_date_key = d.date_key
    JOIN {GOLD_SCHEMA}.dim_company_classification k ON f.classification_key     = k.classification_key
    WHERE f.age_band_key BETWEEN 1 AND 5
),
grouped AS (
    SELECT 'All companies' AS company_group, age_band_key,
           SUM(exposure_years) AS company_years, SUM(deregistrations) AS dereg_all, SUM(deregistrations_excl_batch) AS dereg_excl
    FROM base
    GROUP BY age_band_key
    UNION ALL
    SELECT 'Trading companies', age_band_key,
           SUM(exposure_years), SUM(deregistrations), SUM(deregistrations_excl_batch)
    FROM base
    WHERE borrower_segment = 'Trading company'
    GROUP BY age_band_key
),
scenarios AS (
    SELECT company_group, 1 AS scenario_order, 'All deregistrations'           AS scenario, age_band_key, company_years, dereg_all  AS deregistrations FROM grouped
    UNION ALL
    SELECT company_group, 2,                   'Excluding May-Jun 2026 batch',             age_band_key, company_years, dereg_excl                    FROM grouped
),
calc AS (
    SELECT
        *,
        deregistrations / company_years                                                   AS dereg_rate,
        1 - EXP(-deregistrations / company_years)                                         AS yearly_prob,
        EXP(-SUM(deregistrations / company_years)
             OVER (PARTITION BY company_group, scenario ORDER BY age_band_key))           AS survival
    FROM scenarios
)
SELECT
    c.company_group,
    c.scenario_order,
    c.scenario,
    c.age_band_key                      AS survive_to_year,
    a.age_band_label,
    ROUND(c.company_years, 0)           AS company_years,
    c.deregistrations,
    ROUND(100 * c.dereg_rate,  2)       AS dereg_rate_pct,
    ROUND(100 * c.yearly_prob, 2)       AS yearly_dereg_prob_pct,
    ROUND(100 * c.survival,    1)       AS survival_pct
FROM calc c
JOIN {GOLD_SCHEMA}.dim_age_band a ON c.age_band_key = a.age_band_key

UNION ALL

-- starting point: 100% at registration (year 0)
SELECT DISTINCT
    company_group, scenario_order, scenario,
    0, 'At registration', NULL, NULL, NULL, NULL, 100.0
FROM scenarios

ORDER BY company_group, scenario_order, survive_to_year
"""
df = spark.sql(query)
df.display()

In [0]:
write_report(df, "rpt_survival_curve")

In [0]:
%sql
WITH ordered AS (
    SELECT *,
           LAG(survival_pct) OVER (PARTITION BY company_group, scenario ORDER BY survive_to_year) AS previous_survival
    FROM workspace.sme_risk_gold.rpt_survival_curve
)
SELECT
    COUNT(*)                                                                         AS rows,              
    COUNT_IF(survival_pct > previous_survival)                                       AS survival_increases, 
    MAX(CASE WHEN company_group = 'All companies'     AND scenario_order = 1 AND survive_to_year = 5 THEN survival_pct END) AS all_s5,         
    MAX(CASE WHEN company_group = 'Trading companies' AND scenario_order = 1 AND survive_to_year = 5 THEN survival_pct END) AS trading_s5,     
    MAX(CASE WHEN company_group = 'All companies'     AND scenario_order = 2 AND survive_to_year = 5 THEN survival_pct END) AS all_s5_excl_batch 
FROM ordered

## rpt_segment_risk
Age-adjusted risk by borrower segment (Q2a, compared with all companies) and by warning sign within trading companies (Q2b, compared with all trading companies).
Method: 1 − S(5) from each group's own yearly rates in age bands 0–1 … 4–5. Relative risk index (RRI) = group 1 − S(5) ÷ benchmark 1 − S(5).
Flags: > 1.25 Higher risk, < 0.80 Lower risk. Reliable only if all 5 bands have ≥ 100 company-years and ≥ 10 deregistrations.
Used by the app's "Who is riskiest" page.

In [0]:
query = f"""
-- Step 1: exposure rows for ages 0-5 with the attributes we need
WITH base AS (
    SELECT
        f.age_band_key,
        f.exposure_years,
        f.deregistrations,
        k.borrower_segment,
        k.segment_sort_order,
        c.has_abn,
        c.has_name_change,
        c.is_acn_named
    FROM {GOLD_SCHEMA}.fct_exposure f
    JOIN {GOLD_SCHEMA}.dim_company_classification k ON f.classification_key = k.classification_key
    JOIN {GOLD_SCHEMA}.dim_company c                ON f.company_key        = c.company_key
    WHERE f.age_band_key BETWEEN 1 AND 5
),

-- Step 2: label every row with its analysis and group
labelled AS (
    -- Q2a: borrower segments vs all companies
    SELECT 1 AS analysis_order, 'Borrower segment' AS analysis, 'All companies' AS benchmark,
           0 AS group_order, 'All companies' AS group_name, age_band_key, exposure_years, deregistrations
    FROM base
    UNION ALL
    SELECT 1, 'Borrower segment', 'All companies',
           segment_sort_order, borrower_segment, age_band_key, exposure_years, deregistrations
    FROM base

    -- Q2b: warning signs within trading companies vs all trading companies
    UNION ALL
    SELECT 2, 'Warning sign', 'All trading companies',
           0, 'All trading companies', age_band_key, exposure_years, deregistrations
    FROM base WHERE borrower_segment = 'Trading company'
    UNION ALL
    SELECT 2, 'Warning sign', 'All trading companies',
           CASE WHEN has_abn THEN 1 ELSE 2 END,
           CASE WHEN has_abn THEN 'Has ABN' ELSE 'No ABN' END,
           age_band_key, exposure_years, deregistrations
    FROM base WHERE borrower_segment = 'Trading company'
    UNION ALL
    SELECT 2, 'Warning sign', 'All trading companies',
           CASE WHEN has_name_change THEN 3 ELSE 4 END,
           CASE WHEN has_name_change THEN 'Renamed' ELSE 'Never renamed' END,
           age_band_key, exposure_years, deregistrations
    FROM base WHERE borrower_segment = 'Trading company'
    UNION ALL
    SELECT 2, 'Warning sign', 'All trading companies',
           CASE WHEN is_acn_named THEN 5 ELSE 6 END,
           CASE WHEN is_acn_named THEN 'Numbered (ACN in name)' ELSE 'Named' END,
           age_band_key, exposure_years, deregistrations
    FROM base WHERE borrower_segment = 'Trading company'
),

-- Step 3: totals per group and age band
band_totals AS (
    SELECT analysis_order, analysis, benchmark, group_order, group_name, age_band_key,
           SUM(exposure_years)  AS exposure_years,
           SUM(deregistrations) AS deregistrations
    FROM labelled
    GROUP BY analysis_order, analysis, benchmark, group_order, group_name, age_band_key
),

-- Step 4: 5-year survival per group, plus reliability inputs
group_survival AS (
    SELECT analysis_order, analysis, benchmark, group_order, group_name,
           COUNT(*)                                    AS bands_with_data,
           MIN(exposure_years)                         AS min_band_exposure,
           MIN(deregistrations)                        AS min_band_deregistrations,
           SUM(exposure_years)                         AS exposure_years,
           SUM(deregistrations)                        AS deregistrations,
           EXP(-SUM(deregistrations / exposure_years)) AS survival_5y
    FROM band_totals
    GROUP BY analysis_order, analysis, benchmark, group_order, group_name
),

-- Step 5: risk index vs the benchmark of the same analysis
scored AS (
    SELECT *,
           (1 - survival_5y)
             / MAX(CASE WHEN group_name = benchmark THEN 1 - survival_5y END)
                   OVER (PARTITION BY analysis)                                          AS rri,
           bands_with_data = 5 AND min_band_exposure >= 100
                               AND min_band_deregistrations >= 10                         AS is_reliable
    FROM group_survival
)

SELECT
    analysis_order,
    analysis,
    benchmark,
    group_order,
    group_name,
    group_name = benchmark                          AS is_benchmark,
    ROUND(exposure_years, 0)                        AS company_years_age_0_5,
    deregistrations                                 AS deregistrations_age_0_5,
    ROUND(100 * survival_5y, 1)                     AS survival_5y_pct,
    ROUND(100 * (1 - survival_5y), 1)               AS dereg_within_5y_pct,
    CASE WHEN is_reliable THEN ROUND(rri, 2) END    AS risk_index,
    CASE
        WHEN NOT is_reliable THEN 'Not enough data'
        WHEN rri > 1.25      THEN 'Higher risk'
        WHEN rri < 0.80      THEN 'Lower risk'
        ELSE 'Around average'
    END                                             AS risk_flag,
    is_reliable
FROM scored
ORDER BY analysis_order, group_order
"""
df = spark.sql(query)
df.display()

In [0]:
write_report(df, "rpt_segment_risk")

In [0]:
%sql
SELECT
    COUNT(*)                                                                            AS rows,
    COUNT_IF(is_benchmark AND risk_index <> 1)                                          AS benchmarks_not_1,     
    MAX(CASE WHEN group_name = 'Trading company'        THEN risk_index END)            AS trading_rri,          
    MAX(CASE WHEN group_name = 'Super trustee'          THEN risk_index END)            AS super_trustee_rri,    
    MAX(CASE WHEN group_name = 'Numbered (ACN in name)' THEN risk_index END)            AS numbered_rri,         
    MAX(CASE WHEN group_name = 'Numbered (ACN in name)' THEN dereg_within_5y_pct END)   AS numbered_dereg_5y,    
    MAX(CASE WHEN group_name = 'No ABN'                 THEN risk_index END)            AS no_abn_rri,           
    MAX(CASE WHEN group_name = 'Renamed'                THEN risk_index END)            AS renamed_rri           
FROM workspace.sme_risk_gold.rpt_segment_risk

## rpt_profile_risk
Risk tiers for trading-company borrower profiles (age band × numbered/named × ABN held), from Q3.
Metric: yearly deregistration probability per profile and its multiple of the trading-company average rate, with a 95% interval (rate × e^(±1.96/√d)).
Tiers: High (low end > 2×) · Elevated (low end > 1.25×) · Low (high end < 0.8×) · Standard · Insufficient data (< 100 company-years or < 10 deregistrations).
Tier lift = tier's share of deregistrations ÷ its share of company-years.
Used by the app's "Risk tiers" page and by rpt_company_lookup.

In [0]:
query = f"""
-- Step 1: trading-company exposure with the three profile attributes
WITH trading AS (
    SELECT
        f.age_band_key,
        f.exposure_years,
        f.deregistrations,
        CASE WHEN c.is_acn_named THEN 'Numbered' ELSE 'Named'  END AS name_type,
        CASE WHEN c.has_abn      THEN 'Has ABN'  ELSE 'No ABN' END AS abn_held
    FROM {GOLD_SCHEMA}.fct_exposure f
    JOIN {GOLD_SCHEMA}.dim_company_classification k ON f.classification_key = k.classification_key
    JOIN {GOLD_SCHEMA}.dim_company c                ON f.company_key        = c.company_key
    WHERE k.borrower_segment = 'Trading company'
),

-- Step 2: benchmark = average deregistration rate of all trading companies
benchmark AS (
    SELECT SUM(deregistrations) / SUM(exposure_years) AS bench_rate
    FROM trading
),

-- Step 3: exposure and deregistrations per profile
profiles AS (
    SELECT age_band_key, name_type, abn_held,
           SUM(exposure_years)  AS exposure_years,
           SUM(deregistrations) AS deregistrations
    FROM trading
    GROUP BY age_band_key, name_type, abn_held
),

-- Step 4: trading companies currently in each profile (active, not distressed)
active_now AS (
    SELECT
        s.age_band_key,
        CASE WHEN c.is_acn_named THEN 'Numbered' ELSE 'Named'  END AS name_type,
        CASE WHEN c.has_abn      THEN 'Has ABN'  ELSE 'No ABN' END AS abn_held,
        COUNT(*) AS active_companies
    FROM {GOLD_SCHEMA}.fct_company_snapshot s
    JOIN {GOLD_SCHEMA}.dim_company_classification k ON s.classification_key = k.classification_key
    JOIN {GOLD_SCHEMA}.dim_company c                ON s.company_key        = c.company_key
    WHERE k.borrower_segment = 'Trading company'
      AND s.is_active = 1
      AND s.is_distressed = 0
    GROUP BY 1, 2, 3
),

-- Step 5: rate, probability, multiple and its 95% interval
measured AS (
    SELECT
        p.*,
        a.age_band_label,
        b.bench_rate,
        p.deregistrations / p.exposure_years                                   AS dereg_rate,
        1 - EXP(-p.deregistrations / p.exposure_years)                         AS yearly_prob,
        (p.deregistrations / p.exposure_years) / b.bench_rate                  AS multiple,
        CASE WHEN p.deregistrations > 0
             THEN (p.deregistrations / p.exposure_years) * EXP(-1.96 / SQRT(p.deregistrations)) / b.bench_rate END AS multiple_low,
        CASE WHEN p.deregistrations > 0
             THEN (p.deregistrations / p.exposure_years) * EXP( 1.96 / SQRT(p.deregistrations)) / b.bench_rate END AS multiple_high,
        COALESCE(n.active_companies, 0)                                        AS active_companies
    FROM profiles p
    CROSS JOIN benchmark b
    JOIN {GOLD_SCHEMA}.dim_age_band a ON p.age_band_key = a.age_band_key
    LEFT JOIN active_now n
        ON  p.age_band_key = n.age_band_key
        AND p.name_type    = n.name_type
        AND p.abn_held     = n.abn_held
),

-- Step 6: assign the tier
tiered AS (
    SELECT
        *,
        CASE
            WHEN exposure_years < 100 OR deregistrations < 10 THEN 'Insufficient data'
            WHEN multiple_low  > 2.00 THEN 'High'
            WHEN multiple_low  > 1.25 THEN 'Elevated'
            WHEN multiple_high < 0.80 THEN 'Low'
            ELSE 'Standard'
        END AS risk_tier,
        CASE
            WHEN exposure_years < 100 OR deregistrations < 10 THEN 0
            WHEN multiple_low  > 2.00 THEN 4
            WHEN multiple_low  > 1.25 THEN 3
            WHEN multiple_high < 0.80 THEN 1
            ELSE 2
        END AS tier_order
    FROM measured
),

-- Step 7: each profile's share of the total, and its tier's lift
shares AS (
    SELECT
        *,
        active_companies / SUM(active_companies) OVER ()                        AS share_active,
        exposure_years   / SUM(exposure_years)   OVER ()                        AS share_exposure,
        deregistrations  / SUM(deregistrations)  OVER ()                        AS share_dereg,
        (SUM(deregistrations) OVER (PARTITION BY risk_tier) / SUM(deregistrations) OVER ())
          / (SUM(exposure_years) OVER (PARTITION BY risk_tier) / SUM(exposure_years) OVER ()) AS tier_lift
    FROM tiered
)

SELECT
    tier_order,
    risk_tier,
    age_band_key,
    age_band_label,
    name_type,
    abn_held,
    active_companies,
    ROUND(exposure_years, 0)            AS company_years,
    deregistrations,
    ROUND(100 * yearly_prob, 2)         AS yearly_dereg_prob_pct,
    ROUND(100 * bench_rate, 2)          AS trading_avg_rate_pct,
    ROUND(multiple, 2)                  AS x_trading_average,
    ROUND(multiple_low, 2)              AS x_low_95,
    ROUND(multiple_high, 2)             AS x_high_95,
    ROUND(100 * share_active, 2)        AS pct_of_active_companies,
    ROUND(100 * share_exposure, 2)      AS pct_of_company_years,
    ROUND(100 * share_dereg, 2)         AS pct_of_deregistrations,
    ROUND(tier_lift, 2)                 AS tier_lift
FROM shares
ORDER BY tier_order DESC, multiple DESC
"""
df = spark.sql(query)
df.display()

In [0]:
write_report(df, "rpt_profile_risk")

In [0]:
%sql
SELECT
    risk_tier,
    COUNT(*)                                   AS profiles,
    SUM(active_companies)                      AS active_companies,
    ROUND(SUM(pct_of_active_companies), 1)     AS pct_active,
    ROUND(SUM(pct_of_company_years), 1)        AS pct_company_years,
    ROUND(SUM(pct_of_deregistrations), 1)      AS pct_deregistrations,
    MAX(tier_lift)                             AS tier_lift,
    COUNT(DISTINCT age_band_key || name_type || abn_held) AS distinct_profiles   
FROM workspace.sme_risk_gold.rpt_profile_risk
GROUP BY risk_tier, tier_order
ORDER BY tier_order DESC

## rpt_company_lookup
One row per company with its lending tier, for the app's ACN lookup.
Lending tier (first rule that applies): Excluded (not in population) → Deregistered → Decline / refer (red flag: SOFF or EXAD) → Out of scope (not a trading company) → profile risk tier from rpt_profile_risk (age band × numbered/named × ABN held).
Profile risk = deregistration rates of similar trading companies over the last 12 months, not a prediction for this company.

In [0]:
query = f"""
-- Step 1: one row per company with the attributes the lookup shows
WITH companies AS (
    SELECT
        c.acn,
        c.company_name,
        k.borrower_segment,
        k.company_type_label,
        st.status_code,
        st.status_label,
        c.registration_date,
        dd.full_date                                                   AS deregistration_date,
        f.age_years,
        f.age_band_key,
        a.age_band_label,
        CASE WHEN c.is_acn_named THEN 'Numbered' ELSE 'Named'  END    AS name_type,
        CASE WHEN c.has_abn      THEN 'Has ABN'  ELSE 'No ABN' END    AS abn_held,
        c.abn,
        c.former_name_count,
        c.changed_name_last_12m,
        f.in_population,
        f.is_deregistered,
        f.is_distressed,
        sd.full_date                                                   AS snapshot_date
    FROM {GOLD_SCHEMA}.fct_company_snapshot f
    JOIN {GOLD_SCHEMA}.dim_company c                ON f.company_key        = c.company_key
    JOIN {GOLD_SCHEMA}.dim_company_classification k ON f.classification_key = k.classification_key
    JOIN {GOLD_SCHEMA}.dim_status st                ON f.status_key         = st.status_key
    JOIN {GOLD_SCHEMA}.dim_age_band a               ON f.age_band_key       = a.age_band_key
    JOIN {GOLD_SCHEMA}.dim_date sd                  ON f.snapshot_date_key  = sd.date_key
    LEFT JOIN {GOLD_SCHEMA}.dim_date dd             ON f.deregistration_date_key = dd.date_key
                                                   AND f.deregistration_date_key <> 0
)

-- Step 2: add the profile risk and decide the lending tier
SELECT
    co.acn,
    co.company_name,
    co.borrower_segment,
    co.company_type_label,
    co.status_code,
    co.status_label,
    co.registration_date,
    co.deregistration_date,
    co.age_years,
    co.age_band_label,
    co.name_type,
    co.abn_held,
    co.abn,
    co.former_name_count,
    co.changed_name_last_12m,
    CASE
        WHEN co.in_population = 0                     THEN 'Excluded'
        WHEN co.is_deregistered = 1                   THEN 'Deregistered'
        WHEN co.is_distressed = 1                     THEN 'Decline / refer (red flag)'
        WHEN co.borrower_segment <> 'Trading company' THEN 'Out of scope (not a trading company)'
        ELSE COALESCE(r.risk_tier, 'Insufficient data')
    END                                              AS lending_tier,
    CASE
        WHEN co.in_population = 0                     THEN 0
        WHEN co.is_deregistered = 1                   THEN 7
        WHEN co.is_distressed = 1                     THEN 6
        WHEN co.borrower_segment <> 'Trading company' THEN 0
        ELSE COALESCE(r.tier_order, 0) + 1            -- Low 2 · Standard 3 · Elevated 4 · High 5 · Insufficient 1
    END                                              AS lending_tier_order,
    CASE WHEN co.is_distressed = 1 THEN co.status_label END  AS red_flag,
    r.risk_tier                                      AS profile_risk_tier,
    r.yearly_dereg_prob_pct                          AS profile_yearly_dereg_prob_pct,
    r.x_trading_average                              AS profile_x_trading_average,
    co.snapshot_date
FROM companies co
LEFT JOIN {GOLD_SCHEMA}.rpt_profile_risk r
    ON  co.borrower_segment = 'Trading company'
    AND r.age_band_key = co.age_band_key
    AND r.name_type    = co.name_type
    AND r.abn_held     = co.abn_held
"""
df = spark.sql(query)
df.limit(20).display()      # 4M rows: preview only

In [0]:
write_report(df, "rpt_company_lookup")

# Optional: makes single-ACN lookups faster in the app
spark.sql(f"OPTIMIZE {GOLD_SCHEMA}.rpt_company_lookup ZORDER BY (acn)")

In [0]:
%sql
SELECT
    lending_tier,
    COUNT(*)                                            AS companies,
    COUNT_IF(borrower_segment = 'Trading company')      AS trading_companies
FROM workspace.sme_risk_gold.rpt_company_lookup
GROUP BY lending_tier, lending_tier_order
ORDER BY lending_tier_order DESC

In [0]:
%sql
SELECT
    (SELECT COUNT(*)            FROM workspace.sme_risk_gold.rpt_company_lookup) AS rows,          
    (SELECT COUNT(DISTINCT acn) FROM workspace.sme_risk_gold.rpt_company_lookup) AS distinct_acns, 
    (SELECT COUNT(*)            FROM workspace.sme_risk_gold.dim_company)        AS dim_company_rows;

SELECT * FROM workspace.sme_risk_gold.rpt_company_lookup WHERE acn = '000386685';  

## Hard stops
Fails the notebook (and the job) if any report table is inconsistent with the star schema.

In [0]:
checks = spark.sql(f"""
SELECT
    -- key metrics
    (SELECT COUNT(*) FROM {GOLD_SCHEMA}.rpt_key_metrics) <> 11                                   AS key_metrics_row_count,
    (SELECT COUNT_IF(metric_value IS NULL) FROM {GOLD_SCHEMA}.rpt_key_metrics) > 0                AS key_metrics_nulls,

    -- monthly deregistrations reconcile with the fact table
    (SELECT SUM(deregistrations) FROM {GOLD_SCHEMA}.rpt_monthly_deregistrations)
      <> (SELECT SUM(is_deregistered) FROM {GOLD_SCHEMA}.fct_company_snapshot)                   AS monthly_total_mismatch,

    -- survival can never go up
    (SELECT COUNT(*) FROM (
        SELECT survival_pct,
               LAG(survival_pct) OVER (PARTITION BY company_group, scenario ORDER BY survive_to_year) AS prev
        FROM {GOLD_SCHEMA}.rpt_survival_curve) t
     WHERE survival_pct > prev) > 0                                                              AS survival_increases,

    -- every benchmark has risk index 1
    (SELECT COUNT_IF(is_benchmark AND risk_index <> 1) FROM {GOLD_SCHEMA}.rpt_segment_risk) > 0   AS segment_benchmark_not_1,

    -- profiles: no duplicates, shares add up to 100
    (SELECT COUNT(*) - COUNT(DISTINCT age_band_key, name_type, abn_held)
       FROM {GOLD_SCHEMA}.rpt_profile_risk) > 0                                                  AS profile_duplicates,
    (SELECT ABS(SUM(pct_of_deregistrations) - 100) FROM {GOLD_SCHEMA}.rpt_profile_risk) > 0.5    AS profile_shares_not_100,

    -- lookup: one row per company, every company included
    (SELECT COUNT(*) FROM {GOLD_SCHEMA}.rpt_company_lookup)
      <> (SELECT COUNT(*) FROM {GOLD_SCHEMA}.dim_company)                                        AS lookup_row_mismatch,
    (SELECT COUNT(*) - COUNT(DISTINCT acn) FROM {GOLD_SCHEMA}.rpt_company_lookup) > 0            AS lookup_duplicate_acns,
    (SELECT COUNT_IF(lending_tier IS NULL) FROM {GOLD_SCHEMA}.rpt_company_lookup) > 0            AS lookup_missing_tier
""").first().asDict()

failed = [name for name, is_failed in checks.items() if is_failed]
for name, is_failed in checks.items():
    print(f"{'❌' if is_failed else '✅'} {name}")

if failed:
    raise Exception(f"gold_reports hard stops failed: {failed}")
print(f"\nAll {len(checks)} report checks passed.")